# Chapter 13 — Introducing Automatic Optimization: Let’s Build a Deep Learning Framework

**Nama:** Omar Al Farizy Hutagalung  
**NIM:** 101032300034  
**Mata Kuliah:** Deep Learning

## Tujuan pembelajaran
Bab ini membangun kerangka deep learning sederhana untuk memahami pekerjaan yang biasanya ditangani framework: merekam operasi *forward propagation*, membentuk *computation graph*, menghitung gradien secara otomatis (*autograd*), serta menyusun layer seperti nonlinearity, embedding, cross-entropy, dan recurrent layer.

> Notebook ini merupakan implementasi pembelajaran yang disederhanakan berdasarkan topik Bab 13. Ini bukan salinan kode buku secara verbatim dan tidak dimaksudkan sebagai pengganti seluruh pembahasan buku.


## 1. Apa fungsi deep learning framework?

Framework mengurangi kode berulang dan kompleksitas implementasi. Bagian penting yang dibahas di bab ini adalah:
- **Tensor:** wadah data numerik multidimensi.
- **Computation graph:** hubungan antara tensor hasil operasi dan tensor pembentuknya.
- **Autograd:** menghitung gradien dengan menelusuri graph ke arah belakang.
- **Layer:** abstraksi operasi umum, misalnya nonlinearity, embedding, loss, dan recurrent layer.

Kita mulai dari operasi sederhana agar mekanisme gradien terlihat jelas sebelum menyusun layer yang lebih kompleks.


## 2. Tensor sederhana dengan NumPy
Tensor adalah generalisasi vektor dan matriks: vektor berdimensi satu, matriks berdimensi dua, dan tensor dapat memiliki dimensi lebih tinggi.

In [1]:
import numpy as np

class Tensor:
    def __init__(self, data, creators=None, creation_op=None):
        self.data = np.array(data, dtype=float)
        self.creators = creators
        self.creation_op = creation_op
        self.grad = None

    def __add__(self, other):
        return Tensor(
            self.data + other.data,
            creators=[self, other],
            creation_op="add"
        )

    def __repr__(self):
        return f"Tensor({self.data!r})"

x = Tensor([1, 2, 3, 4, 5])
y = Tensor([2, 2, 2, 2, 2])
z = x + y

print("x =", x.data)
print("y =", y.data)
print("z = x + y =", z.data)

x = [1. 2. 3. 4. 5.]
y = [2. 2. 2. 2. 2.]
z = x + y = [3. 4. 5. 6. 7.]


## 3. Membuat computation graph dan autograd untuk penjumlahan

Setiap tensor hasil menyimpan `creators` (tensor yang membentuknya) dan `creation_op` (operasi pembentuknya). Saat gradien pada hasil penjumlahan ditelusuri mundur, gradien diteruskan ke kedua input karena turunan penjumlahan terhadap masing-masing input adalah 1.

Implementasi di bawah sengaja hanya mendukung operasi `add`, seperti tahap awal penjelasan buku. Ini belum merupakan autograd umum untuk semua operasi.

In [2]:
class AddTensor:
    def __init__(self, data, creators=None, creation_op=None):
        self.data = np.array(data, dtype=float)
        self.creators = creators
        self.creation_op = creation_op
        self.grad = None

    def __add__(self, other):
        return AddTensor(self.data + other.data,
                         creators=[self, other],
                         creation_op="add")

    def backward(self, grad):
        grad = np.array(grad, dtype=float)
        self.grad = grad if self.grad is None else self.grad + grad
        if self.creation_op == "add":
            self.creators[0].backward(grad)
            self.creators[1].backward(grad)

a = AddTensor([1, 2, 3])
b = AddTensor([10, 20, 30])
c = a + b
c.backward([1, 1, 1])

print("c =", c.data)
print("gradien c =", c.grad)
print("gradien a =", a.grad)
print("gradien b =", b.grad)

c = [11. 22. 33.]
gradien c = [1. 1. 1.]
gradien a = [1. 1. 1.]
gradien b = [1. 1. 1.]


### 3.1 Mengapa gradien diteruskan seperti itu?

Untuk operasi element-wise \(z=x+y\), berlaku ∂z/∂x = 1 dan ∂z/∂y = 1. Dengan aturan rantai, gradien dari `z` diteruskan ke `x` dan `y`. Bila sebuah tensor berkontribusi ke hasil melalui lebih dari satu jalur, gradien yang diterima dari setiap jalur harus dijumlahkan. Contoh berikut menunjukkan jalur berulang.

In [3]:
a = AddTensor([1.0, 2.0])
b = AddTensor([3.0, 4.0])
c = a + b
d = c + a               
d.backward([1.0, 1.0])

print("d =", d.data)
print("gradien a (dua jalur) =", a.grad)
print("gradien b =", b.grad)

d = [5. 8.]
gradien a (dua jalur) = [2. 2.]
gradien b = [1. 1.]


## 4. Nonlinearity layer

Jaringan saraf memerlukan fungsi nonlinier agar tidak hanya menjadi transformasi linear bertumpuk. Bab ini membahas layer nonlinearity sebagai salah satu komponen framework. Contoh berikut memperlihatkan fungsi tanh dan turunannya secara numerik; ini adalah demonstrasi konsep, bukan implementasi lengkap framework.

In [4]:
def tanh_forward(x):
    return np.tanh(x)

def tanh_backward(x, grad_output):
    # d(tanh(x))/dx = 1 - tanh(x)^2
    return grad_output * (1.0 - np.tanh(x) ** 2)

x = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
y = tanh_forward(x)
grad = tanh_backward(x, np.ones_like(x))
print("input :", x)
print("tanh  :", np.round(y, 4))
print("gradien lokal:", np.round(grad, 4))

input : [-2. -1.  0.  1.  2.]
tanh  : [-0.964  -0.7616  0.      0.7616  0.964 ]
gradien lokal: [0.0707 0.42   1.     0.42   0.0707]


## 5. Embedding layer

Embedding memetakan ID token diskret ke vektor berdimensi tetap. Matriks embedding memiliki bentuk `(jumlah_token, dimensi_embedding)`. Untuk sebuah token, forward pass memilih baris yang sesuai. Dalam pelatihan, hanya baris embedding yang dipakai token tersebut yang menerima kontribusi gradien.

Contoh berikut menggunakan bobot acak agar mekanismenya mudah diamati; bobot ini belum dilatih dan bukan hasil numerik dari buku.

In [5]:
rng = np.random.default_rng(13)
vocab = {"the": 0, "cat": 1, "sat": 2, "dog": 3}
embedding_dim = 4
embedding_matrix = rng.normal(0, 0.1, size=(len(vocab), embedding_dim))

tokens = ["the", "cat", "sat"]
token_ids = np.array([vocab[t] for t in tokens])
embedded = embedding_matrix[token_ids]

print("token IDs:", token_ids)
print("bentuk matriks embedding:", embedding_matrix.shape)
print("bentuk hasil embedding:", embedded.shape)
print(embedded)

token IDs: [0 1 2]
bentuk matriks embedding: (4, 4)
bentuk hasil embedding: (3, 4)
[[ 0.18267566 -0.30783319  0.0958064   0.00696372]
 [ 0.131825    0.03856292  0.18272586  0.00317438]
 [-0.05162294  0.05804849  0.04321069 -0.03568394]]


## 6. Cross-entropy loss

Untuk klasifikasi, model menghasilkan skor/logit per kelas. Softmax mengubah logits menjadi distribusi probabilitas, dan cross-entropy mengukur seberapa jauh prediksi dari label target. Nilai loss lebih kecil ketika probabilitas untuk kelas target lebih besar.

In [6]:
def softmax(logits):
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=-1, keepdims=True)

def cross_entropy(logits, target_index):
    probs = softmax(np.asarray(logits, dtype=float))
    return -np.log(probs[target_index] + 1e-12)

logits = np.array([1.2, 0.3, -0.2])
target_index = 0
probabilities = softmax(logits)
loss = cross_entropy(logits, target_index)

print("probabilitas kelas:", np.round(probabilities, 4))
print("kelas target:", target_index)
print("cross-entropy:", round(float(loss), 4))

probabilitas kelas: [0.6049 0.2459 0.1492]
kelas target: 0
cross-entropy: 0.5027


## 7. Recurrent layer: memproses urutan

Recurrent neural network memproses masukan berurutan sambil membawa *hidden state* dari satu langkah waktu ke langkah berikutnya. Bentuk umum yang sering digunakan untuk menggambarkan ide ini adalah:

\[
h_t = f(x_t, h_{t-1})
\]

State pada waktu \(t\) bergantung pada input saat ini dan state sebelumnya. Contoh minimal berikut menunjukkan alur state; parameter sengaja dibuat sederhana untuk demonstrasi.

In [7]:
def simple_recurrent_forward(sequence, input_weights, recurrent_weights, bias):
    hidden = np.zeros(recurrent_weights.shape[0])
    states = []
    for x_t in sequence:
        hidden = np.tanh(input_weights @ x_t + recurrent_weights @ hidden + bias)
        states.append(hidden.copy())
    return np.array(states)

sequence = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
Wx = np.array([[0.4, -0.2], [0.1, 0.3]])
Wh = np.array([[0.2, 0.1], [-0.1, 0.25]])
bias = np.zeros(2)

states = simple_recurrent_forward(sequence, Wx, Wh, bias)
print("hidden state setiap langkah:")
print(np.round(states, 4))

hidden state setiap langkah:
[[ 0.3799  0.0997]
 [-0.1136  0.2793]
 [ 0.2024  0.4472]]


## 8. Rangkuman dan refleksi

- Framework membantu mengotomatisasi backpropagation dan pembaruan parameter.
- Tensor menyimpan data; computation graph menyimpan hubungan operasi.
- Autograd menelusuri graph untuk meneruskan gradien.
- Layer membungkus operasi yang sering digunakan, termasuk nonlinearity, embedding, loss, dan recurrent layer.
- Implementasi dalam notebook ini hanya contoh edukatif. Autograd-nya belum mendukung operasi selain penjumlahan dan tidak dapat langsung dipakai untuk melatih seluruh jaringan di atas.

### Pertanyaan refleksi
1. Apa fungsi `creators` dan `creation_op` pada tensor?
2. Mengapa gradien untuk operasi penjumlahan diteruskan ke kedua input?
3. Mengapa gradien dari beberapa jalur harus dijumlahkan?
4. Apa perbedaan tensor biasa dengan computation graph?
5. Mengapa framework yang lebih lengkap memerlukan aturan backward untuk setiap operasi/layer?
